In [8]:
import torch
import whisper
import soundfile as sf
from pyannote.audio import Pipeline
from dotenv import load_dotenv
import os

load_dotenv()
token = os.getenv("HF_TOKEN")

AUDIO  = r"C:\Users\lucasmg-cogna\Dev\AI-Audio-Transcription-Examples\audio\simulacao-de-dialogo.wav"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# whisper
model  = whisper.load_model("large-v3-turbo", device=device)
result = model.transcribe(AUDIO, language="pt", verbose=False, fp16=True)

# pré-carrega o áudio — bypassa o AudioDecoder/torchcodec
waveform, sample_rate = sf.read(AUDIO, dtype="float32")
waveform = torch.tensor(waveform).T.unsqueeze(0)  # (1, channels, time) ou (1, time)

# diarization
pipeline    = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-community-1",
    token=token
).to(device)

diarization = pipeline(
    {"waveform": waveform, "sample_rate": sample_rate},
    num_speakers=2
)

# merge
for seg in result["segments"]:
    mid     = (seg["start"] + seg["end"]) / 2
    speaker = "desconhecido"
    for turn, _, spk in diarization.speaker_diarization.itertracks(yield_label=True):
        if turn.start <= mid <= turn.end:
            speaker = spk
            break
    print(f"[{speaker}] [{seg['start']:.2f}s → {seg['end']:.2f}s] {seg['text'].strip()}")

100%|██████████| 7943/7943 [00:04<00:00, 1788.27frames/s]
c:\Users\lucasmg-cogna\Dev\AI-Audio-Transcription-Examples\.venv\Lib\site-packages\pyannote\audio\models\blocks\pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\ReduceOps.cpp:1857.)
  std = sequences.std(dim=-1, correction=1)


[SPEAKER_01] [0.00s → 3.26s] Oi, tudo bem? Me chamo Lana, sou aluna aqui da Anguera.
[SPEAKER_01] [3.62s → 6.28s] Queria saber como faço para me inscrever no vestibular.
[SPEAKER_00] [6.90s → 9.54s] Oi, Lana. Tudo ótimo e com você.
[SPEAKER_00] [10.16s → 12.02s] Sou o Lucas, atendente aqui da unidade.
[SPEAKER_00] [12.70s → 14.12s] Posso te ajudar com isso sim.
[SPEAKER_00] [14.72s → 18.00s] Já tem algum curso que te interessa ou ainda está na dúvida?
[SPEAKER_01] [18.44s → 22.38s] Bom, então eu já sei o que quero, ciência de dados, o tecnólogo.
[SPEAKER_01] [22.72s → 25.92s] Vi que tem muito a ver com inteligência artificial, é isso mesmo?
[SPEAKER_00] [26.48s → 30.00s] Exatamente. É um curso bem completo nessa área.
[SPEAKER_00] [30.58s → 35.94s] Você vai ver desde análise e tratamento de dados até machine learning e visualização de informações.
[SPEAKER_00] [36.98s → 38.76s] Está super em alta no mercado agora.
[desconhecido] [39.32s → 43.96s] Você já tem alguma afinidade com a área